1. Загрузите данные из файла data-logistic.csv. Это двумерная выборка, целевая переменная на которой принимает значения -1 или 1.

In [1]:
import numpy as np
import pandas as pd
from sklearn.metrics import roc_auc_score

data = pd.read_csv('data-logistic.csv', header=None)
X = np.array(data.drop(0,axis=1))  
y = np.array(data[0])  


2. Реализуйте градиентный спуск для обычной и L2-регуляризованной (с коэффициентом регуляции 10) логистической регрессии. Используйте длину шага k=0.1. В качестве начального приближения используйте вектор (0, 0).

In [44]:

k = 0.1    
max_iter = 10000
eps = 10**(-5)

def compute_gradient(w, X, y, C):
    ell = len(y)
    M = y * (w[0] * X[:, 0] + w[1] * X[:, 1])
    coeff = 1.0 - 1.0 / (1.0 + np.exp(-M))
    coeff_stable = 1.0 / (1.0 + np.exp(M))
    grad1 = np.mean(y * X[:, 0] * coeff_stable)
    grad2 = np.mean(y * X[:, 1] * coeff_stable)
    grad1 -= C * w[0]
    grad2 -= C * w[1]
    return np.array([grad1, grad2])

def gradient_descent(X, y, C, k, max_iter, tol, start_w=np.array([0.0, 0.0])):
    w = start_w
    for i in range(max_iter):
        grad = compute_gradient(w, X, y, C)
        w_new = w + k * grad  
        if np.linalg.norm(w_new - w) <= tol:
            w = w_new
            break
        w = w_new
    return (w, i)

3. Запустите градиентный спуск и доведите до сходимости (евклидово расстояние между векторами весов на соседних итерациях должно быть не больше 1e-5).

In [45]:
w_no_reg, steps_no_reg = gradient_descent(X, y, C=0.0, k=k, max_iter=max_iter, tol=eps)
w_reg, steps_reg = gradient_descent(X, y, C=10.0, k=k, max_iter=max_iter, tol=eps)

4. Какое значение принимает AUC-ROC на обучении без регуляризации и при ее использовании?

In [46]:
def sigmoid(x):
    return 1.0 / (1.0 + np.exp(-x))

scores_no_reg = sigmoid(w_no_reg[0] * X[:, 0] + w_no_reg[1] * X[:, 1])
auc_no_reg = roc_auc_score(y, scores_no_reg)
scores_reg = sigmoid(w_reg[0] * X[:, 0] + w_reg[1] * X[:, 1])
auc_reg = roc_auc_score(y, scores_reg)
print('auc_score без регуляризации и с ней:',round(auc_no_reg, 3), round(auc_reg,3))
print('число шагов без регуляризации и с ней:',steps_no_reg, steps_reg)

auc_score без регуляризации и с ней: 0.927 0.936
число шагов без регуляризации и с ней: 243 7


5. Попробуйте поменять длину шага. Будет ли сходиться алгоритм, если делать более длинные шаги?

In [47]:
w_no_reg, steps_no_reg = gradient_descent(X, y, C=0.0, k=0.01, max_iter=max_iter, tol=eps)
w_reg, steps_reg = gradient_descent(X, y, C=10.0, k=0.01, max_iter=max_iter, tol=eps)
scores_no_reg = sigmoid(w_no_reg[0] * X[:, 0] + w_no_reg[1] * X[:, 1])
auc_no_reg = roc_auc_score(y, scores_no_reg)
scores_reg = sigmoid(w_reg[0] * X[:, 0] + w_reg[1] * X[:, 1])
auc_reg = roc_auc_score(y, scores_reg)
print('k = 0.01')
print('auc_score без регуляризации и с ней:',round(auc_no_reg, 3), round(auc_reg,3))
print('число шагов без регуляризации и с ней:',steps_no_reg, steps_reg)

w_no_reg, steps_no_reg = gradient_descent(X, y, C=0.0, k=0.2, max_iter=max_iter, tol=eps)
w_reg, steps_reg = gradient_descent(X, y, C=10.0, k=0.2, max_iter=max_iter, tol=eps)
scores_no_reg = sigmoid(w_no_reg[0] * X[:, 0] + w_no_reg[1] * X[:, 1])
auc_no_reg = roc_auc_score(y, scores_no_reg)
scores_reg = sigmoid(w_reg[0] * X[:, 0] + w_reg[1] * X[:, 1])
auc_reg = roc_auc_score(y, scores_reg)
print('k = 0.2')
print('auc_score без регуляризации и с ней:',round(auc_no_reg, 3), round(auc_reg,3))
print('число шагов без регуляризации и с ней:',steps_no_reg, steps_reg)
print('при k > 0.2 расходится алгоритм (выйдет ошибка)')

k = 0.01
auc_score без регуляризации и с ней: 0.927 0.936
число шагов без регуляризации и с ней: 1478 46


C:\Users\User\AppData\Local\Temp\ipykernel_11608\2243636369.py:8: RuntimeWarning: overflow encountered in exp
  coeff = 1.0 - 1.0 / (1.0 + np.exp(-M))
C:\Users\User\AppData\Local\Temp\ipykernel_11608\2243636369.py:9: RuntimeWarning: overflow encountered in exp
  coeff_stable = 1.0 / (1.0 + np.exp(M))


k = 0.2
auc_score без регуляризации и с ней: 0.927 0.246
число шагов без регуляризации и с ней: 134 9999
при k > 0.2 расходится алгоритм


C:\Users\User\AppData\Local\Temp\ipykernel_11608\3856135716.py:2: RuntimeWarning: overflow encountered in exp
  return 1.0 / (1.0 + np.exp(-x))


6. Попробуйте менять начальное приближение.

In [53]:
w_no_reg, steps_no_reg = gradient_descent(X, y, C=0.0, k=0.1, max_iter=max_iter, tol=eps, start_w=np.array([-10000.0, 10000.0]))
w_reg, steps_reg = gradient_descent(X, y, C=10.0, k=0.1, max_iter=max_iter, tol=eps, start_w=np.array([-10000.0, 10000.0]))
scores_no_reg = sigmoid(w_no_reg[0] * X[:, 0] + w_no_reg[1] * X[:, 1])
auc_no_reg = roc_auc_score(y, scores_no_reg)
scores_reg = sigmoid(w_reg[0] * X[:, 0] + w_reg[1] * X[:, 1])
auc_reg = roc_auc_score(y, scores_reg)
print('start_w = [-10000.0, 10000.0]')
print('auc_score без регуляризации и с ней:',round(auc_no_reg, 3), round(auc_reg,3))
print('число шагов без регуляризации и с ней:',steps_no_reg, steps_reg)
print('Мы упираемся в локальные минимумы без регуляризации')
print('Длина шага ни на что особо не влияет')

C:\Users\User\AppData\Local\Temp\ipykernel_11608\2243636369.py:8: RuntimeWarning: overflow encountered in exp
  coeff = 1.0 - 1.0 / (1.0 + np.exp(-M))
C:\Users\User\AppData\Local\Temp\ipykernel_11608\2243636369.py:9: RuntimeWarning: overflow encountered in exp
  coeff_stable = 1.0 / (1.0 + np.exp(M))
C:\Users\User\AppData\Local\Temp\ipykernel_11608\3856135716.py:2: RuntimeWarning: overflow encountered in exp
  return 1.0 / (1.0 + np.exp(-x))


start_w = [-10000.0, 10000.0]
auc_score без регуляризации и с ней: 0.522 0.936
число шагов без регуляризации и с ней: 9999 7
Мы упираемся в локальные минимумы без регуляризации
Длина шага ни на что особо не влияет
